## **Imports & path**

In [2]:
import json
import random
from pathlib import Path
from PIL import Image
import numpy as np
from collections import Counter
from datasets import load_dataset
import pandas as pd

random.seed(42)

DATA = Path("..") / "data"
SPLITS_DIR = DATA / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# U-Net paths
SEG_IMG = DATA / "kvasir-seg" / "images"
SEG_MASK = DATA / "kvasir-seg" / "masks"
NORMAL_DIR = DATA / "kvasir-v2-normal"
NORMAL_CLASSES = ["normal-pylorus", "normal-z-line"]
NORMAL_MASK_DIR = NORMAL_DIR / "masks"

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **I. U-Net**

### **Tạo mask cho ảnh bình thường của dataset Kvasir-V2**

In [3]:
created = 0
skipped = 0

for cls in NORMAL_CLASSES:
    img_dir = NORMAL_DIR / cls
    mask_dir = NORMAL_MASK_DIR / cls
    mask_dir.mkdir(parents=True, exist_ok=True)

    for img_path in sorted(img_dir.glob("*.jpg")):
        mask_path = mask_dir / img_path.name
        if mask_path.exists():
            skipped += 1
            continue

        img = Image.open(img_path)
        w, h = img.size
        mask = Image.fromarray(np.zeros((h, w), dtype=np.uint8))
        mask.save(mask_path, format="JPEG")
        created += 1

print(f"Created: {created} | Skipped: {skipped}")

Created: 0 | Skipped: 2000


In [4]:
for cls in NORMAL_CLASSES:
    imgs = sorted((NORMAL_DIR / cls).glob("*.jpg"))
    masks = sorted((NORMAL_MASK_DIR / cls).glob("*.jpg"))
    print(f"{cls}: {len(imgs)} images, {len(masks)} masks")

    # Kiểm tra kích thước khớp nhau (sample 3 ảnh)
    for img_path in random.sample(imgs, min(3, len(imgs))):
        mask_path = NORMAL_MASK_DIR / cls / img_path.name
        img = Image.open(img_path)
        mask = Image.open(mask_path)
        match = img.size == mask.size
        print(f"  {img_path.name}: img {img.size} | mask {mask.size} | {'OK' if match else 'MISMATCH'}")

normal-pylorus: 1000 images, 1000 masks
  c01809cc-864f-40cd-af58-5f718f0cd0dd.jpg: img (1280, 1024) | mask (1280, 1024) | OK
  262cd554-00d6-4e40-85db-4a8b9bb1e486.jpg: img (720, 576) | mask (720, 576) | OK
  07bbcfc5-2f5e-41c4-a8e0-fc7bef0439cd.jpg: img (1280, 1024) | mask (1280, 1024) | OK
normal-z-line: 1000 images, 1000 masks
  d071cc90-4f5b-4134-bba9-ffc613d5a001.jpg: img (1280, 1024) | mask (1280, 1024) | OK
  59e83b73-70d5-4ee1-b591-d35cde6b9972.jpg: img (1280, 1024) | mask (1280, 1024) | OK
  4cf7535b-588e-4286-afb1-ca4863ed9dbd.jpg: img (1280, 1024) | mask (1280, 1024) | OK


### **Split data**

In [5]:
# Polyp: 800/100/100
polyp_files = sorted(SEG_IMG.glob("*.jpg"))
random.shuffle(polyp_files)
polyp_train = polyp_files[:800]
polyp_val = polyp_files[800:900]
polyp_test = polyp_files[900:]

# Normal: mỗi class 360/45/45
normal_splits = {"train": [], "val": [], "test": []}
for cls in NORMAL_CLASSES:
    cls_files = sorted((NORMAL_DIR / cls).glob("*.jpg"))
    random.shuffle(cls_files)
    normal_splits["train"].extend([(f, cls) for f in cls_files[:360]])
    normal_splits["val"].extend([(f, cls) for f in cls_files[360:405]])
    normal_splits["test"].extend([(f, cls) for f in cls_files[405:450]])

def make_entry(img_path, mask_path, label):
    return {
        "image": str(img_path.relative_to(DATA)),
        "mask": str(mask_path.relative_to(DATA)),
        "label": label
    }

unet_split = {}
for split_name, polyp_list in [("train", polyp_train), ("val", polyp_val), ("test", polyp_test)]:
    entries = []
    # Polyp
    for p in polyp_list:
        entries.append(make_entry(p, SEG_MASK / p.name, "polyp"))
    # Normal
    for f, cls in normal_splits[split_name]:
        entries.append(make_entry(f, NORMAL_MASK_DIR / cls / f.name, "normal"))
    unet_split[split_name] = entries

# Save
unet_path = SPLITS_DIR / "unet_split.json"
with open(unet_path, "w") as f:
    json.dump(unet_split, f, indent=2)

print(f"Saved: {unet_path}")

Saved: ..\data\splits\unet_split.json


In [6]:
for split_name, entries in unet_split.items():
    labels = Counter(e["label"] for e in entries)
    print(f"{split_name:5s}: {len(entries)} total | polyp: {labels['polyp']} | normal: {labels['normal']}")

# Kiểm tra không trùng ảnh giữa các split
all_images = {}
for split_name, entries in unet_split.items():
    for e in entries:
        img = e["image"]
        if img in all_images:
            print(f"TRÙNG: {img} trong {all_images[img]} và {split_name}")
        all_images[img] = split_name
print(f"\nTổng ảnh unique: {len(all_images)} (expected: {sum(len(v) for v in unet_split.values())})")

train: 1520 total | polyp: 800 | normal: 720
val  : 190 total | polyp: 100 | normal: 90
test : 190 total | polyp: 100 | normal: 90

Tổng ảnh unique: 1900 (expected: 1900)


## **II. BLIP**

### **Load và gộp x1 (train và test)**

In [7]:
ds = load_dataset("SimulaMet/Kvasir-VQA-x1")
train_df = pd.DataFrame(ds["train"])
test_df = pd.DataFrame(ds["test"])

# Gộp train + test
all_df = pd.concat([train_df, test_df], ignore_index=True)
# Bỏ cột original
all_df = all_df.drop(columns=["original"])

print(f"Train gốc: {len(train_df):,} QA | Test gốc: {len(test_df):,} QA")
print(f"Gộp lại  : {len(all_df):,} QA | Unique img_id: {all_df['img_id'].nunique()}")

Train gốc: 143,594 QA | Test gốc: 15,955 QA
Gộp lại  : 159,549 QA | Unique img_id: 6449


### **Split theo img_id (80/10/10)**

In [8]:
unique_ids = list(all_df["img_id"].unique())
random.shuffle(unique_ids)

n = len(unique_ids)
n_train = int(n * 0.8)
n_val = int(n * 0.1)

train_ids = set(unique_ids[:n_train])
val_ids = set(unique_ids[n_train:n_train + n_val])
test_ids = set(unique_ids[n_train + n_val:])

print(f"img_id split: train {len(train_ids)} | val {len(val_ids)} | test {len(test_ids)}")
print(f"Total: {len(train_ids) + len(val_ids) + len(test_ids)} (expected: {n})")

img_id split: train 5159 | val 644 | test 646
Total: 6449 (expected: 6449)


In [9]:
def build_entries(df, id_set):
    subset = df[df["img_id"].isin(id_set)]
    entries = []
    for _, row in subset.iterrows():
        qclass = row["question_class"]
        if isinstance(qclass, np.ndarray):
            qclass = qclass.tolist()
        entries.append({
            "img_id": row["img_id"],
            "image": f"kvasir-vqa/images/{row['img_id']}.jpg",
            "question": row["question"],
            "answer": row["answer"],
            "complexity": int(row["complexity"]),
            "question_class": qclass
        })
    return entries

blip_split = {
    "train": build_entries(all_df, train_ids),
    "val": build_entries(all_df, val_ids),
    "test": build_entries(all_df, test_ids),
}

blip_path = SPLITS_DIR / "blip_split.json"
with open(blip_path, "w", encoding="utf-8") as f:
    json.dump(blip_split, f, indent=2, ensure_ascii=False)

print(f"Saved: {blip_path}")
for s, entries in blip_split.items():
    img_count = len(set(e["img_id"] for e in entries))
    print(f"  {s:5s}: {len(entries):,} QA pairs | {img_count} images")

Saved: ..\data\splits\blip_split.json
  train: 128,494 QA pairs | 5159 images
  val  : 16,095 QA pairs | 644 images
  test : 14,960 QA pairs | 646 images


## **III. Phân bố data sau khi split lại (BLIP)**

In [10]:
for split_name, entries in blip_split.items():
    comps = Counter(e["complexity"] for e in entries)
    total = sum(comps.values())
    print(f"{split_name:5s}:", end="")
    for level in sorted(comps):
        print(f"  complexity {level}: {comps[level]/total*100:.1f}%", end="")
    print()

train:  complexity 1: 34.4%  complexity 2: 32.8%  complexity 3: 32.8%
val  :  complexity 1: 34.4%  complexity 2: 32.8%  complexity 3: 32.8%
test :  complexity 1: 34.5%  complexity 2: 32.7%  complexity 3: 32.7%


In [11]:
def get_class_dist(entries):
    classes = []
    for e in entries:
        for c in e["question_class"]:
            classes.append(c)
    total = len(classes)
    return {k: v / total * 100 for k, v in Counter(classes).items()}

train_dist = get_class_dist(blip_split["train"])
val_dist = get_class_dist(blip_split["val"])
test_dist = get_class_dist(blip_split["test"])

all_classes = sorted(set(train_dist) | set(val_dist) | set(test_dist))

print(f"{'Class':<25s} {'Train%':>8s} {'Val%':>8s} {'Test%':>8s} {'Max Diff':>10s}")
print("-" * 65)
alert = False
for cls in all_classes:
    t = train_dist.get(cls, 0)
    v = val_dist.get(cls, 0)
    te = test_dist.get(cls, 0)
    diff = max(abs(t - v), abs(t - te), abs(v - te))
    flag = " Cảnh báo" if diff > 5 else ""
    print(f"{cls:<25s} {t:>7.2f}% {v:>7.2f}% {te:>7.2f}% {diff:>9.2f}%{flag}")
    if diff > 5:
        alert = True

if alert:
    print("\n Có class chênh > 5%. Cân nhắc đổi seed và split lại.")
else:
    print("\nOK — tất cả class chênh ≤ 5%.")

Class                       Train%     Val%    Test%   Max Diff
-----------------------------------------------------------------
abnormality_color            5.91%    5.80%    5.71%      0.19%
abnormality_location         5.97%    5.81%    5.80%      0.17%
abnormality_presence         5.88%    5.70%    5.79%      0.18%
box_artifact_presence        7.43%    7.23%    7.52%      0.29%
finding_count                6.48%    6.52%    6.58%      0.10%
finding_presence             0.78%    0.77%    0.90%      0.14%
instrument_count             6.66%    6.98%    6.96%      0.33%
instrument_location          5.93%    6.21%    6.08%      0.28%
instrument_presence          5.93%    6.18%    6.00%      0.25%
landmark_color               0.17%    0.14%    0.15%      0.03%
landmark_location            4.16%    4.24%    3.97%      0.27%
landmark_presence            3.79%    4.08%    3.57%      0.51%
polyp_count                  5.98%    5.83%    5.74%      0.24%
polyp_removal_status         7.46%    

In [12]:
for split_name, entries in blip_split.items():
    count = sum(1 for e in entries if "landmark_color" in e["question_class"])
    img_count = len(set(e["img_id"] for e in entries if "landmark_color" in e["question_class"]))
    print(f"{split_name:5s}: {count} QA pairs ({img_count} images) chứa landmark_color")

train: 445 QA pairs (73 images) chứa landmark_color
val  : 46 QA pairs (8 images) chứa landmark_color
test : 43 QA pairs (7 images) chứa landmark_color
